# Bronze to Silver Transformation

## Import Libraries

In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, 1c3784b7-75b0-4aa8-baaa-70613ef15792, 3, Finished, Available, Finished, False)

## Read Bronze Tables

In [2]:
customers_df = spark.table("bronze_customers")
campaigns_df = spark.table("bronze_campaigns")
customer_events_df = spark.table("bronze_customer_events")
products_df = spark.table("bronze_products")
support_cases_df = spark.table("bronze_support_cases")

StatementMeta(, 1c3784b7-75b0-4aa8-baaa-70613ef15792, 4, Finished, Available, Finished, False)

## Validate Bronze Data

In [4]:
display(customers_df)

StatementMeta(, 1c3784b7-75b0-4aa8-baaa-70613ef15792, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ac92d851-e51e-4f72-88da-e9775497e313)

In [5]:
display(campaigns_df)

StatementMeta(, 1c3784b7-75b0-4aa8-baaa-70613ef15792, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a1b9a30d-0b4c-4874-98ec-2f82af796667)

In [6]:
display(customer_events_df)


StatementMeta(, 1c3784b7-75b0-4aa8-baaa-70613ef15792, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ef53cfb1-dffa-400e-957e-1dd584095171)

In [7]:
display(products_df)

StatementMeta(, 1c3784b7-75b0-4aa8-baaa-70613ef15792, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6599a23d-fc91-4298-b91a-6684e2e6cd2c)

In [8]:
display(support_cases_df)

StatementMeta(, 1c3784b7-75b0-4aa8-baaa-70613ef15792, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 366f7dc6-773c-4bcf-8868-3320150cbcef)

## Apply Silver Transformations

In [10]:
from pyspark.sql.functions import col, trim, upper, initcap, to_date, to_timestamp, when

# Silver Customers
silver_customers = (
    customers_df
    .dropDuplicates(["customer_id"])
    .withColumn("first_name", initcap(trim(col("first_name"))))
    .withColumn("last_name", initcap(trim(col("last_name"))))
    .withColumn("gender", initcap(trim(col("gender"))))
    .withColumn("city", initcap(trim(col("city"))))
    .withColumn("state", initcap(trim(col("state"))))
    .withColumn("country", initcap(trim(col("country"))))
    .withColumn("registration_date", to_date(col("registration_date")))
    .filter(col("customer_id").isNotNull())
)

# Silver Products
silver_products = (
    products_df
    .dropDuplicates(["product_id"])
    .withColumn("product_name", initcap(trim(col("product_name"))))
    .withColumn("category", initcap(trim(col("category"))))
    .withColumn("brand", initcap(trim(col("brand"))))
    .withColumn("price", col("price").cast("double"))
    .withColumn("stock_quantity", col("stock_quantity").cast("int"))
    .filter(col("product_id").isNotNull())
)

# Silver Campaigns
silver_campaigns = (
    campaigns_df
    .dropDuplicates(["campaign_id"])
    .withColumn("campaign_name", initcap(trim(col("campaign_name"))))
    .withColumn("channel", initcap(trim(col("channel"))))
    .withColumn("start_date", to_date(col("start_date")))
    .withColumn("end_date", to_date(col("end_date")))
    .withColumn("budget", col("budget").cast("double"))
    .filter(col("campaign_id").isNotNull())
)

# Silver Support Cases
silver_support_cases = (
    support_cases_df
    .dropDuplicates(["ticket_id"])
    .withColumn("issue_type", initcap(trim(col("issue_type"))))
    .withColumn("priority", initcap(trim(col("priority"))))
    .withColumn("status", initcap(trim(col("status"))))
    .withColumn("created_date", to_date(col("created_date")))
    .withColumn("resolved_date", to_date(col("resolved_date")))
    .filter(col("ticket_id").isNotNull())
)

# Silver Customer Events
silver_customer_events = (
    customer_events_df
    .dropDuplicates(["event_id"])
    .withColumn("event_timestamp", to_timestamp(col("event_timestamp")))
    .withColumn("event_type", lower(trim(col("event_type"))))
    .withColumn("channel", initcap(trim(col("channel"))))
    .withColumn("device", initcap(trim(col("device"))))
    .withColumn("event_value", col("event_value").cast("double"))
    .filter(col("event_id").isNotNull())
    .filter(col("customer_id").isNotNull())
)

print("Silver transformations completed successfully")

StatementMeta(, 1c3784b7-75b0-4aa8-baaa-70613ef15792, 12, Finished, Available, Finished, False)

Silver transformations completed successfully


## Write Silver Delta Tables

In [11]:
silver_customers.write.mode("overwrite").format("delta").saveAsTable("silver_customers")
silver_products.write.mode("overwrite").format("delta").saveAsTable("silver_products")
silver_campaigns.write.mode("overwrite").format("delta").saveAsTable("silver_campaigns")
silver_support_cases.write.mode("overwrite").format("delta").saveAsTable("silver_support_cases")
silver_customer_events.write.mode("overwrite").format("delta").saveAsTable("silver_customer_events")

print("Silver tables created successfully")

StatementMeta(, 1c3784b7-75b0-4aa8-baaa-70613ef15792, 13, Finished, Available, Finished, False)

Silver tables created successfully
